# Audio Preprocessing
---
Handles: noise reduction, silence trimming, normalization, and audio cleanup.

## Importing Audio
---
Load audio file & check if they are good to go.

In [ ]:
import librosa
import librosa.feature
import librosa.beat
import librosa.onset
import os

def load_and_validate_audio_pair(user_path, ref_path, sr=22050):

    # Step 1: Check files exist
    print("\n[1/4] Checking files...")

    if not os.path.exists(user_path):
        print(f"❌ User audio not found: {user_path}")
        return None, None, sr, False, "User audio file not found"

    if not os.path.exists(ref_path):
        print(f"❌ Reference audio not found: {ref_path}")
        return None, None, sr, False, "Reference audio file not found"

    print("✓ Both files found")

    # Step 2: Load audio
    print("\n[2/4] Loading audio...")

    try:
        user_audio, sr = librosa.load(user_path, sr=sr)
        print(f"✓ User audio: {len(user_audio)/sr:.2f}s ({len(user_audio)} samples)")
    except Exception as e:
        print(f"❌ Failed to load user audio: {e}")
        return None, None, sr, False, f"Could not load user audio: {e}"

    try:
        ref_audio, sr = librosa.load(ref_path, sr=sr)
        print(f"✓ Reference audio: {len(ref_audio)/sr:.2f}s ({len(ref_audio)} samples)")
    except Exception as e:
        print(f"❌ Failed to load reference audio: {e}")
        return None, None, sr, False, f"Could not load reference audio: {e}"

    # Step 3: Check duration
    print("\n[3/4] Checking duration...")

    user_duration = len(user_audio) / sr
    ref_duration = len(ref_audio) / sr
    min_duration = 2.0

    if user_duration < min_duration:
        error = f"User audio too short: {user_duration:.2f}s (min: {min_duration}s)"
        print(f"❌ {error}")
        return None, None, sr, False, error

    if ref_duration < min_duration:
        error = f"Reference audio too short: {ref_duration:.2f}s (min: {min_duration}s)"
        print(f"❌ {error}")
        return None, None, sr, False, error

    print(f"✓ Both audios long enough")

    # Step 4: Check audio content (not silent)
    print("\n[4/4] Checking audio content...")

    user_rms = np.sqrt(np.mean(user_audio ** 2))
    ref_rms = np.sqrt(np.mean(ref_audio ** 2))

    if user_rms < 0.001:
        error = f"User audio is too quiet (RMS: {user_rms:.6f})"
        print(f"❌ {error}")
        return None, None, sr, False, error

    if ref_rms < 0.001:
        error = f"Reference audio is too quiet (RMS: {ref_rms:.6f})"
        print(f"❌ {error}")
        return None, None, sr, False, error

    user_peak = np.max(np.abs(user_audio))
    ref_peak = np.max(np.abs(ref_audio))

    print(f"✓ User: RMS={user_rms:.4f}, Peak={user_peak:.4f}")
    print(f"✓ Reference: RMS={ref_rms:.4f}, Peak={ref_peak:.4f}")

    # All checks passed!
    print("\n✓ ALL VALIDATION CHECKS PASSED!")

    return user_audio, ref_audio, sr, True, ""

## Silence Trimming
---
Remove silence from beginning and end of audio.

In [ ]:
import numpy as np

def trim_silence(audio, sr, threshold_db=-40):
    print("Trimming silence...")

    # Create spectrogram and convert to dB
    S = librosa.feature.melspectrogram(y=audio, sr=sr)
    S_db = librosa.power_to_db(S, ref=np.max)

    # Get energy per frame
    energy = np.mean(S_db, axis=0)

    # Find frames above threshold (-40 dB soft singing)
    active_frames = energy > threshold_db
    active_indices = np.where(active_frames)[0]

    if len(active_indices) == 0:
        return audio

    # Get start and end frames
    start_frame = active_indices[0]
    end_frame = active_indices[-1]

    # Convert to samples
    start_sample = librosa.frames_to_samples(start_frame, hop_length=512)
    end_sample = librosa.frames_to_samples(end_frame, hop_length=512)

    # Add 100ms buffer for smooth transitions
    buffer = int(0.1 * sr)
    start_sample = max(0, start_sample - buffer)
    end_sample = min(len(audio), end_sample + buffer)

    # Trim and return
    audio_trimmed = audio[start_sample:end_sample]

    print(f"✓ Trimmed silence!")
    print(f"  Original: {len(audio)/sr:.2f}s")
    print(f"  Trimmed:  {len(audio_trimmed)/sr:.2f}s")
    print(f"  Removed:  {(len(audio) - len(audio_trimmed))/sr:.2f}s of silence")

    return audio_trimmed

## Noise Reduction
---
Reduce background noise using spectral subtraction.

In [ ]:
# Noise reduction function
def reduce_noise_spectral_subtraction(audio, sr, noise_duration=1.0):
    print("Reducing noise...")

    # Get spectrogram
    S = librosa.feature.melspectrogram(y=audio, sr=sr)

    # Extract noise profile
    noise_frame_count = int(sr * noise_duration / 512)
    noise_sample = S[:, :noise_frame_count]
    noise_profile = np.mean(noise_sample, axis=1, keepdims=True)

    # Subtract noise profile
    S_reduced = S - 2.0 * noise_profile # noise_factor = 2 is the factor which can change aggressiveness of reduced audio
    S_reduced = np.maximum(S_reduced, 0)

    # Convert back to audio
    audio_reduced = librosa.feature.inverse.mel_to_audio(S_reduced, sr=sr)

    # Normalize to original level
    # Spectral noise reduction alters signal amplitude and dynamic range.
    # To ensure consistent feature extraction, especially RMS energy and
    # emotion-related features, I re-scaled the processed signal to match
    # the original peak amplitude.
    max_original = np.max(np.abs(audio))
    max_reduced = np.max(np.abs(audio_reduced))

    if max_reduced > 1e-8:
        audio_reduced *= (max_original / max_reduced) # so that (red_max = org_max)

    print(f"✓ Noise reduction complete!")

    return audio_reduced

## Audio Normalization
---
Normalize audio to target loudness level (in dB).

In [ ]:
# normalization of audio so that it suits each type of singer (loud or quiet)
def normalize_loudness(audio, target_level_db=-3.0):
    # target = -3 because
    # Digital audio maximum amplitude = 0 dB
    # so if audio crosses 0dB it is clipped but for singing spikes are natural
    # therefore for safe level we normalized according to -3dB

    # Calculate current RMS (loudness)
    rms = np.sqrt(np.mean(audio ** 2))

    if rms == 0:
        return audio

    # Convert to dB
    current_level_db = 20 * np.log10(rms + 1e-10)

    # Calculate gain needed
    gain_db = target_level_db - current_level_db
    gain_linear = 10 ** (gain_db / 20) # dB to linear conversion

    # Apply gain
    audio_normalized = audio * gain_linear

    # Prevent clipping becoz librosa digital limit is (-1 to +1 or 0dB)
    max_val = np.max(np.abs(audio_normalized))
    if max_val > 1.0:
        audio_normalized = audio_normalized / max_val

    print(f"Normalized audio: {current_level_db:.1f} dB -> {target_level_db:.1f} dB")

    return audio_normalized

---
### Preprocessing complete
---

# Segment Matching
---
Handles: auto-matching user audio to reference song, confidence scoring, manual fallback.

## Validating singing reference Audio
---
Checks: sufficient pitched content

In [ ]:
def validate_pitched_content(audio):

    try:
        # Extract pitch
        f0,voiced_flags, voiced_probs = librosa.pyin(audio, fmin=50, fmax=2000)

        # Count voiced frames
        voiced_frames = np.sum(voiced_probs > 0.1)
        total_frames = len(f0)

        voiced_percentage = 100 * voiced_frames / total_frames

        has_content = voiced_percentage > 5  # At least 5% voiced

        return has_content, voiced_percentage

    except Exception as e:
        print(f"Warning: Could not analyze pitched content: {e}")
        return True, 0

## Auto match
---
Calculation of DTW cost (the less the better)

Find best matching segment in reference song using optimized two-pass approach.

    Pass 1: Fast correlation-based filtering (find top 3 candidates)
    Pass 2: Accurate DTW on top candidates (find best match)

In [ ]:
def compute_dtw_cost(X,Y):
    n, m = len(X), len(Y)

    # Initialize cost matrix
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0

    # Fill matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Euclidean distance between frames
            cost = np.linalg.norm(X[i-1] - Y[j-1])

            # Recurrence relation: take minimum of three previous cells
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],      # insertion
                dtw_matrix[i, j-1],      # deletion
                dtw_matrix[i-1, j-1]     # match
            )

    # Normalize by sequence length because n+m doesnt matter for big data, returns small values
    return dtw_matrix[n, m] / (n + m)

In [ ]:
def find_best_matching_segment(user_audio, ref_audio,sr,window_size):

    if window_size is None:
        # Add 20% buffer to user length
        window_size = int(len(user_audio) * 1.2)

    print(f"User audio: {len(user_audio)/sr:.1f}s, Window size: {window_size/sr:.1f}s")
    print(f"Reference: {len(ref_audio)/sr:.1f}s")

    # ===== PASS 1: Quick energy correlation filter =====
    print("PASS 1: Fast correlation filtering...")

    # user_energy gives singing pattern of the user, through calculating loudness of each frame
    user_energy = librosa.feature.rms(y=user_audio)[0]

    candidates = []  # List of (correlation_score, start_sample)

    # Slide window with 1-second steps (fast)
    step_samples = sr

    for start in range(0, len(ref_audio) - window_size, step_samples):
        ref_segment = ref_audio[start:start + window_size]
        ref_energy = librosa.feature.rms(y=ref_segment)[0]

        # Match energy lengths
        if len(ref_energy) > len(user_energy):
            ref_energy = ref_energy[:len(user_energy)]
        elif len(ref_energy) < len(user_energy): # not possible because window size =1.2 * user_audio,applied for just in cases
            continue  # Skip if ref segment is too short

        # Correlation of energy profiles from correlation matrix (2x2)
        correlation = np.corrcoef(user_energy, ref_energy)[0, 1]

        if not np.isnan(correlation):
            candidates.append((correlation, start))

    # Sort by correlation (highest first)
    candidates.sort(reverse=True)

    print(f"Found {len(candidates)} candidate windows")

    if len(candidates) == 0:
        print("Warning: No candidates found. Using first segment.")
        return ref_audio[:window_size], 0.0, 0.0

    # ===== PASS 2: Accurate DTW on top 3 candidates =====
    print("PASS 2: Accurate DTW on top candidates...")

    # Extract MFCC for user (once)
    mfcc_user = librosa.feature.mfcc(y=user_audio, sr=sr, n_mfcc=13)

    best_cost = np.inf
    best_start = 0

    # Only check top 3 candidates
    for rank, (corr, start) in enumerate(candidates[:3]):
        print(f"  Candidate {rank+1}: correlation={corr:.3f}, position={start/sr:.1f}s")

        ref_segment = ref_audio[start:start + window_size]
        mfcc_ref = librosa.feature.mfcc(y=ref_segment, sr=sr, n_mfcc=13)

        # DTW cost
        cost = compute_dtw_cost(mfcc_user.T, mfcc_ref.T)
        print(f"    DTW cost: {cost:.4f}")

        if cost < best_cost:
            best_cost = cost
            best_start = start

    timestamp = best_start / sr
    matched_segment = ref_audio[best_start:best_start + window_size]

    print()
    print(f"Best match at {timestamp:.1f}s (DTW cost: {best_cost:.4f})")
    print("=" * 50)

    # Calculate confidence
    confidence = max(0, min(100, 100 * (1 - best_cost / 20)))

    return matched_segment, timestamp, confidence


## Prepare User audio
---

Prepare audio pair for comparison by handling length mismatches, after ref audio is converted into matched segment.

In [ ]:
def prepare_for_comparison(user_audio, ref_audio,sr):
    user_duration = len(user_audio) / sr
    ref_duration = len(ref_audio) / sr

    print(f"Audio lengths: user={user_duration:.1f}s, ref={ref_duration:.1f}s")

    # Case 1: User audio much shorter than reference (< 50%)
    if user_duration < 0.5 * ref_duration:
        target_length = int(len(user_audio) * 1.2)
        ref_audio = ref_audio[:target_length]
        print(f"Trimmed reference to {len(ref_audio)/sr:.1f}s to match user")

    # Case 2: User audio much longer than reference (> 150%)
    elif user_duration > 1.5 * ref_duration:
        print(f"Warning: User provided {user_duration:.1f}s, reference is {ref_duration:.1f}s")
        print("Note: User recording may contain multiple repetitions")

    return user_audio, ref_audio

# Feature Extraction - Pitch
---



In [ ]:
# Pitch class mapping
PITCH_CLASS_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

## Pitch extraction
---

Extract pitch contour using PYIN algorithm.

In [ ]:
def extract_pitch_contour(audio, sr,fmin = 50, fmax =2000):

    print(f"Extracting pitch contour (fmin={fmin}, fmax={fmax})...")

    # PYIN returns (f0, voiced_flag, voiced_probs)
    f0, voiced_flag, voiced_probs = librosa.pyin(
        audio,
        fmin=fmin,
        fmax=fmax,
        frame_length=2048
    )

    valid_frames = np.sum(~np.isnan(f0))
    voiced_frames = np.sum(voiced_probs > 0.1)

    print(f"Extracted {len(f0)} frames ({len(f0) * 512 / sr:.2f}s)")
    print(f"Valid frames: {valid_frames}, Voiced frames: {voiced_frames}")

    return f0, voiced_probs

## Key detection & Transpose
---
Estimate musical key from pitch contour.<br>
Uses median pitch and maps to nearest pitch class.

Transpose pitch contour by N semitones.
<br>Uses equal temperament formula : `f' = f * 2^(semitones/12)`

In [ ]:
def detect_key_from_pitch(f0, voiced_probs):
    # Filter valid, voiced frames
    valid_idx = (~np.isnan(f0)) & (voiced_probs > 0.1)
    f0_valid = f0[valid_idx]

    if len(f0_valid) < 10:
        print("Warning: Insufficient voiced frames for key detection")
        return 0  # Default to C

    # Median pitch
    median_f0 = np.median(f0_valid)

    # Convert to MIDI note number (A4=440Hz = note 69)
    midi_note = 12 * np.log2(median_f0 / 440) + 69

    # Get pitch class (0-11)
    pitch_class = int(np.round(midi_note)) % 12

    print(f"Median pitch: {median_f0:.1f} Hz -> Estimated key: {PITCH_CLASS_NAMES[pitch_class]}")

    return pitch_class

In [ ]:
def transpose_pitch(f0, semitones):
    # Formula: f' = f * 2^(semitones/12)
    factor = 2 ** (semitones / 12)
    f0_transposed = f0 * factor

    print(f"Transposed pitch by {semitones:+d} semitones (factor: {factor:.4f})")

    return f0_transposed

## Handling Key shifts
---
Transpose user pitch to match reference key, according to detected key shift.

In [ ]:
def handle_key_shift(f0_user, voiced_user, f0_ref, voiced_ref):
    # Detect keys from pitch contours
    key_user = detect_key_from_pitch(f0_user, voiced_user)
    key_ref = detect_key_from_pitch(f0_ref, voiced_ref)

    # Calculate offset
    semitone_offset = (key_user - key_ref) % 12

    # Prefer smaller offset
    if semitone_offset > 6:
        semitone_offset -= 12

    print(f"Key offset: {semitone_offset:+d} semitones ({PITCH_CLASS_NAMES[key_user]} -> {PITCH_CLASS_NAMES[key_ref]})")

    # Transpose user pitch to match reference key
    f0_user_corrected = transpose_pitch(f0_user, -semitone_offset)

    return f0_user_corrected, semitone_offset